# Electivo de Bioinformática — Clase 2

## Ambiente Linux y Bash (nivel intermedio)

**Programa:** Doctorado — 2º año
**Duración:** 3 horas
**Fecha:** 11 de agosto

### Objetivos de la clase

Al finalizar esta clase, serán capaces de:

1. Distinguir entre variables de shell y variables de entorno, y usar `PATH`, `alias` y `~/.bashrc`.
2. Usar comodines (`*`, `?`, `{}`) para trabajar con múltiples archivos a la vez.
3. Escribir, dar permisos y ejecutar scripts de bash simples.
4. Usar argumentos posicionales y variables especiales dentro de un script.
5. Implementar control de flujo (`if`/`elif`/`else`) y bucles (`for`/`while`).
6. Escribir funciones de bash reutilizables.
7. Procesar archivos de texto tabulares con `sed` y `awk`.
8. Comprimir y empaquetar archivos con `tar` y `gzip`.

### Estructura de la sesión (3 horas)

| Bloque | Tema | Tiempo aprox. |
|---|---|---|
| 1 | Repaso de la clase 1 y objetivos de hoy | 10 min |
| 2 | Variables de shell vs. entorno, `PATH`, `alias`, `~/.bashrc` | 25 min |
| — | *Pausa* | 10 min |
| 3 | Comodines y expansión de nombres (`glob`) | 15 min |
| 4 | Scripts de bash: shebang, permisos, estructura | 15 min |
| 5 | Argumentos posicionales y variables especiales | 20 min |
| 6 | Control de flujo: `test` e `if`/`elif`/`else` | 25 min |
| 7 | Bucles: `for` y `while` | 25 min |
| 8 | Funciones en bash | 15 min |
| 9 | Introducción a `sed` y `awk` | 20 min |
| 10 | Compresión y archivado: `tar` y `gzip` | 10 min |
| 11 | Ejercicio integrador | 25 min |
| 12 | Cierre, resumen y tarea | 5 min |

> **Nota sobre este notebook:** al igual que en la Clase 1, las celdas de código usan la magia `%%bash` y se ejecutan localmente para que puedan practicar sin depender de la conexión al clúster. Todo lo que veamos hoy funciona igual una vez conectados por SSH al servidor de la UDD — de hecho, les recomendamos repetir estos mismos comandos en su sesión SSH durante la semana.
>
> **Importante:** cada celda `%%bash` se ejecuta en una *shell nueva e independiente*. Esto significa que variables, `alias` o el directorio de trabajo (`cd`) definidos en una celda **no se mantienen automáticamente** en la siguiente, a menos que se vuelvan a definir o se guarden en un archivo (como `~/.bashrc`). En una sesión SSH normal esto no ocurre, porque toda la terminal es una sola shell continua.

---
## 1. Repaso de la clase 1

En la clase anterior vimos:

- Qué es un clúster y cómo conectarse vía **SSH**.
- Navegación básica: `pwd`, `ls`, `cd`.
- Gestión de archivos y directorios: `mkdir`, `touch`, `cp`, `mv`, `rm`.
- Lectura de archivos: `cat`, `less`, `head`, `tail`, `wc`.
- Búsqueda: `grep`, `find`.
- Redirección y tuberías: `>`, `>>`, `|`.
- Permisos (`chmod`) y una primera mirada a procesos (`ps`) y variables de entorno (`$HOME`, `$PATH`).

Hoy profundizamos en **cómo se configura y personaliza el ambiente de trabajo en Linux**, y damos el salto de "escribir comandos sueltos" a **escribir scripts de bash**: programas que combinan variables, condicionales y bucles para automatizar tareas repetitivas — algo esencial en bioinformática, donde procesamos decenas o cientos de muestras con la misma lógica.

In [1]:
%%bash
# Recreamos el espacio de trabajo de la clase 1 y creamos uno nuevo para hoy
mkdir -p ~/curso_bioinfo/clase2
cd ~/curso_bioinfo/clase2
pwd
ls -la

/root/curso_bioinfo/clase2
total 8
drwxr-xr-x 2 root root 4096 Aug 10 15:45 .
drwxr-xr-x 3 root root 4096 Aug 10 15:45 ..


---
## 2. Variables de shell vs. variables de entorno, `PATH`, `alias` y `~/.bashrc`

### 2.1 Variables de shell

Se definen así (**sin espacios** alrededor del `=`) y se leen con `$`:

```bash
nombre="valor"
echo "$nombre"
```

Por defecto, una variable de shell **solo existe en la shell actual**: no la heredan los programas ni las subshells que se lancen desde ahí.

### 2.2 Variables de entorno

Si usamos `export`, la variable pasa a formar parte del **entorno**, y sí es heredada por cualquier proceso hijo (otro script, otro programa) que se lance desde esa shell:

```bash
export nombre="valor"
```

Convención: las variables de entorno suelen escribirse en MAYÚSCULAS (`$HOME`, `$PATH`, `$USER`), aunque esto es solo una convención, no una regla del lenguaje.

In [2]:
%%bash
MI_VAR="hola clase"
echo "Variable local en esta shell: $MI_VAR"

echo "--- Intentando verla desde una subshell (sin export) ---"
bash -c 'echo "Desde subshell: [$MI_VAR]"'

echo "--- Ahora la exportamos ---"
export MI_VAR
bash -c 'echo "Desde subshell: [$MI_VAR]"'

Variable local en esta shell: hola clase
--- Intentando verla desde una subshell (sin export) ---
Desde subshell: []
--- Ahora la exportamos ---
Desde subshell: [hola clase]


Noten la diferencia: sin `export`, la subshell no ve la variable (queda vacía). Con `export`, sí la ve. Esto es clave cuando lanzan scripts o programas externos (por ejemplo, un pipeline que llama a otros scripts) que necesitan leer alguna variable que ustedes definieron.

### 2.3 `PATH`: dónde busca el sistema los ejecutables

`$PATH` es una variable de entorno con una lista de directorios (separados por `:`) donde el sistema busca el programa cuando escriben un comando.

In [4]:
%%bash
echo "$PATH" | tr ':' '\n'

echo "--- ¿Dónde está el ejecutable de python3? ---"
which python3

echo "--- ¿Es un comando interno (builtin) o un ejecutable externo? ---"
type cd
type ls

/opt/bin
/usr/local/sbin
/usr/local/bin
/usr/sbin
/usr/bin
/sbin
/bin
/tools/node/bin
/tools/google-cloud-sdk/bin
--- ¿Dónde está el ejecutable de python3? ---
/usr/bin/python3
--- ¿Es un comando interno (builtin) o un ejecutable externo? ---
cd is a shell builtin
ls is /usr/bin/ls


Si alguna vez instalan un programa y el sistema responde `command not found`, casi siempre es porque el directorio donde quedó instalado el programa no está incluido en `$PATH`. La solución típica es agregarlo en `~/.bashrc`:

```bash
export PATH="$HOME/mis_programas/bin:$PATH"
```

### 2.4 `alias`: atajos para comandos frecuentes

Un `alias` reemplaza un nombre corto por un comando (o combinación de comandos) más largo.

> Nota técnica: bash solo expande alias en shells **interactivas** por defecto. En una terminal normal esto no es problema, pero en un script (o en esta celda `%%bash`, que no es interactiva) hay que activarlo explícitamente con `shopt -s expand_aliases`. Por eso lo verán en la celda siguiente — en su terminal SSH normal no lo necesitan.

In [5]:
%%bash
shopt -s expand_aliases

alias ll='ls -la'
alias grep='grep --color=auto'

echo "--- Alias definidos en esta shell ---"
alias

echo "--- Usando el alias ---"
cd ~/curso_bioinfo/clase2
ll

--- Alias definidos en esta shell ---
alias grep='grep --color=auto'
alias ll='ls -la'
--- Usando el alias ---
total 8
drwxr-xr-x 2 root root 4096 Aug 10 15:45 .
drwxr-xr-x 3 root root 4096 Aug 10 15:45 ..


### 2.5 `~/.bashrc`: personalizando la shell de forma persistente

`~/.bashrc` es un script que bash ejecuta automáticamente **cada vez que se abre una nueva shell interactiva**. Ahí es donde se guardan de forma permanente las variables de entorno, `alias` y configuraciones que quieran tener disponibles siempre (por ejemplo, cada vez que se conectan por SSH al clúster).

Como cada celda de este notebook es una shell nueva, no podemos "ver" el efecto persistente aquí directamente, pero sí podemos escribir en el archivo y confirmar que quedó guardado:

In [6]:
%%bash
cat >> ~/.bashrc << 'EOF'

# --- Configuración agregada en la Clase 2 ---
export CURSO_BIOINFO="$HOME/curso_bioinfo"
alias ll='ls -la'
alias cdcurso='cd $CURSO_BIOINFO'
EOF

echo "--- Últimas líneas de ~/.bashrc ---"
tail -n 6 ~/.bashrc

--- Últimas líneas de ~/.bashrc ---


# --- Configuración agregada en la Clase 2 ---
export CURSO_BIOINFO="$HOME/curso_bioinfo"
alias ll='ls -la'
alias cdcurso='cd $CURSO_BIOINFO'


En una sesión SSH real, después de editar `~/.bashrc` deben ejecutar:

```bash
source ~/.bashrc
```

(o abrir una terminal nueva) para que los cambios tomen efecto en la sesión actual, ya que el archivo solo se lee automáticamente al **iniciar** una shell.

---
## 3. Comodines y expansión de nombres (`glob`)

Los comodines permiten referirse a **varios archivos a la vez** sin escribir sus nombres uno por uno — algo constante en bioinformática, donde se procesan lotes de archivos (`muestra1.fastq`, `muestra2.fastq`, …).

| Patrón | Significa |
|---|---|
| `*` | Cero o más caracteres cualquiera |
| `?` | Exactamente un caracter cualquiera |
| `[abc]` | Un caracter entre los indicados |
| `[0-9]` | Un caracter dentro del rango |
| `{fastq,txt}` | Cualquiera de las alternativas listadas |

Esto se llama *globbing* y lo interpreta la propia shell **antes** de pasarle los argumentos al comando (no es una funcionalidad de `ls`, `cp`, etc. — funciona igual con cualquier comando).

In [8]:
%%bash
mkdir -p ~/curso_bioinfo/clase2/wildcards
cd ~/curso_bioinfo/clase2/wildcards

touch muestra1.fastq muestra2.fastq muestra10.fastq control.txt notas.md

echo "--- Todos los .fastq ---"
ls *.fastq

echo "--- 'muestra' + un solo caracter + .fastq (NO incluye muestra10) ---"
ls muestra?.fastq

echo "--- Archivos .fastq o .txt ---"
ls *.{fastq,txt}

--- Todos los .fastq ---
muestra10.fastq
muestra1.fastq
muestra2.fastq
--- 'muestra' + un solo caracter + .fastq (NO incluye muestra10) ---
muestra1.fastq
muestra2.fastq
--- Archivos .fastq o .txt ---
control.txt
muestra10.fastq
muestra1.fastq
muestra2.fastq


---
## 4. Scripts de bash: shebang, permisos y estructura

Un **script de bash** es simplemente un archivo de texto con una secuencia de comandos, que puede ejecutarse como un programa.

### Elementos clave

1. **Shebang** (primera línea): le indica al sistema qué intérprete usar.
   ```bash
   #!/bin/bash
   ```
2. **Comentarios**: cualquier línea que empieza con `#` (excepto el shebang).
3. **Permiso de ejecución**: con `chmod +x script.sh`.
4. **Ejecución**: `./script.sh` (si tiene permiso de ejecución y está en el directorio actual) o `bash script.sh` (no requiere permiso de ejecución).

### Buena práctica: `set -euo pipefail`

Al inicio de un script, esta línea hace que:

- `-e`: el script se detenga inmediatamente si un comando falla.
- `-u`: se detenga si se usa una variable no definida.
- `-o pipefail`: una tubería (`|`) falle si **cualquiera** de sus comandos falla (no solo el último).

Esto evita que errores silenciosos pasen desapercibidos en medio de un análisis largo.

In [9]:
%%bash
cd ~/curso_bioinfo/clase2

cat > saludo.sh << 'EOF'
#!/bin/bash
set -euo pipefail

# Script simple de saludo
echo "Hola desde un script de bash"
echo "Hoy es: $(date)"
echo "Estoy corriendo en: $(hostname)"
EOF

echo "--- Permisos antes ---"
ls -l saludo.sh

chmod +x saludo.sh

echo "--- Permisos después de chmod +x ---"
ls -l saludo.sh

echo "--- Ejecutando ---"
./saludo.sh

--- Permisos antes ---
-rw-r--r-- 1 root root 155 Aug 10 16:54 saludo.sh
--- Permisos después de chmod +x ---
-rwxr-xr-x 1 root root 155 Aug 10 16:54 saludo.sh
--- Ejecutando ---
Hola desde un script de bash
Hoy es: Mon Aug 10 04:54:48 PM UTC 2026
Estoy corriendo en: e2247bb82d27


---
## 5. Argumentos posicionales y variables especiales

Un script puede recibir argumentos desde la línea de comandos, igual que cualquier otro programa de Linux.

| Variable | Significado |
|---|---|
| `$0` | Nombre del script |
| `$1`, `$2`, … | Primer, segundo, … argumento |
| `$#` | Número total de argumentos |
| `$@` | Todos los argumentos, como lista separada |
| `$?` | Código de salida del último comando (`0` = éxito, distinto de `0` = error) |

In [10]:
%%bash
cd ~/curso_bioinfo/clase2

cat > procesar_muestra.sh << 'EOF'
#!/bin/bash
set -euo pipefail

echo "Nombre del script: $0"
echo "Primer argumento (nombre de muestra): $1"
echo "Segundo argumento (umbral de calidad): $2"
echo "Numero de argumentos recibidos: $#"
echo "Todos los argumentos: $@"
EOF
chmod +x procesar_muestra.sh

./procesar_muestra.sh muestra1.fastq 30
echo "--- Codigo de salida del comando anterior: $? ---"

echo "--- Que pasa si falta un argumento? (set -u lo detecta) ---"
./procesar_muestra.sh muestra1.fastq || echo "El script fallo como se esperaba, codigo de salida: $?"

Nombre del script: ./procesar_muestra.sh
Primer argumento (nombre de muestra): muestra1.fastq
Segundo argumento (umbral de calidad): 30
Numero de argumentos recibidos: 2
Todos los argumentos: muestra1.fastq 30
--- Codigo de salida del comando anterior: 0 ---
--- Que pasa si falta un argumento? (set -u lo detecta) ---
Nombre del script: ./procesar_muestra.sh
Primer argumento (nombre de muestra): muestra1.fastq
El script fallo como se esperaba, codigo de salida: 1


./procesar_muestra.sh: line 6: $2: unbound variable


---
## 6. Control de flujo: `test` e `if` / `elif` / `else`

### Sintaxis básica

```bash
if [ condición ]; then
    comandos
elif [ otra_condición ]; then
    comandos
else
    comandos
fi
```

`[ condición ]` es en realidad el comando `test`. Comparadores más usados:

| Tipo | Operador | Significado |
|---|---|---|
| Numérico | `-eq`, `-ne`, `-lt`, `-le`, `-gt`, `-ge` | igual, distinto, menor, menor o igual, mayor, mayor o igual |
| Texto | `=`, `!=`, `-z`, `-n` | igual, distinto, cadena vacía, cadena no vacía |
| Archivos | `-f`, `-d`, `-e` | es archivo regular, es directorio, existe |

> Tip: `[[ condición ]]` (doble corchete) es una versión más moderna y flexible de `test`, disponible en bash (aunque no en `sh` puro). Para este curso, `[ ]` es suficiente y más portable.

In [ ]:
%%bash
cd ~/curso_bioinfo/clase2

# Generamos un archivo de prueba con "lecturas" simuladas
printf "@read1\nACGTACGTAC\n+\nIIIIIIIIII\n@read2\nTTGGCCAATT\n+\nIIIIIIIIII\n" > muestra1.fastq

cat > chequeo_calidad.sh << 'EOF'
#!/bin/bash
set -euo pipefail

archivo=$1
umbral=$2

if [ ! -f "$archivo" ]; then
    echo "Error: el archivo '$archivo' no existe"
    exit 1
fi

lineas=$(wc -l < "$archivo")

if [ "$lineas" -ge "$umbral" ]; then
    echo "$archivo tiene $lineas lineas (>= $umbral): OK"
elif [ "$lineas" -ge 1 ]; then
    echo "$archivo tiene $lineas lineas (< $umbral): revisar, pocas lecturas"
else
    echo "$archivo esta vacio"
fi
EOF
chmod +x chequeo_calidad.sh

echo "--- Caso 1: archivo existe y cumple el umbral ---"
./chequeo_calidad.sh muestra1.fastq 4

echo "--- Caso 2: archivo existe pero no cumple el umbral ---"
./chequeo_calidad.sh muestra1.fastq 100

echo "--- Caso 3: archivo no existe ---"
./chequeo_calidad.sh no_existe.fastq 4 || echo "Termino con codigo de salida $?"

--- Caso 1: archivo existe y cumple el umbral ---


muestra1.fastq tiene 8 lineas (>= 4): OK


--- Caso 2: archivo existe pero no cumple el umbral ---


muestra1.fastq tiene 8 lineas (< 100): revisar, pocas lecturas


--- Caso 3: archivo no existe ---


Error: el archivo 'no_existe.fastq' no existe


Termino con codigo de salida 1


---
## 7. Bucles: `for` y `while`

### `for`: iterar sobre una lista de valores (muy usado con comodines)

```bash
for variable in lista; do
    comandos
done
```

### `while`: repetir mientras una condición sea verdadera

```bash
while [ condición ]; do
    comandos
done
```

### `while read`: leer un archivo línea por línea (patrón muy común en bioinformática, por ejemplo para recorrer una lista de muestras)

```bash
while read -r linea; do
    comandos
done < archivo.txt
```

In [ ]:
%%bash
cd ~/curso_bioinfo/clase2
mkdir -p datos && cd datos

# Generamos 3 archivos fastq simulados
for i in 1 2 3; do
    printf "@read${i}_a\nACGTACGTAC\n+\nIIIIIIIIII\n@read${i}_b\nTTGGCCAATT\n+\nIIIIIIIIII\n" > muestra${i}.fastq
done
ls

echo "--- for sobre los archivos generados (con comodin) ---"
for f in *.fastq; do
    n=$(grep -c "^@" "$f")
    echo "$f: $n lecturas"
done

muestra1.fastq
muestra2.fastq
muestra3.fastq


--- for sobre los archivos generados (con comodin) ---


muestra1.fastq: 2 lecturas


muestra2.fastq: 2 lecturas


muestra3.fastq: 2 lecturas


In [ ]:
%%bash
cd ~/curso_bioinfo/clase2/datos

echo "--- while con contador ---"
contador=1
while [ "$contador" -le 3 ]; do
    echo "Iteracion numero $contador"
    contador=$((contador + 1))
done

echo "--- while read: procesando una lista de muestras desde un archivo ---"
cat > ../lista_muestras.txt << 'EOF'
muestra1
muestra2
muestra3
EOF

while read -r muestra; do
    echo "Procesando: ${muestra}.fastq"
done < ../lista_muestras.txt

--- while con contador ---


Iteracion numero 1


Iteracion numero 2
Iteracion numero 3
--- while read: procesando una lista de muestras desde un arch

ivo ---


Procesando: muestra1.fastq
Procesando: muestra2.fastq
Procesando: muestra3.fastq


---
## 8. Funciones en bash

Una función agrupa comandos bajo un nombre reutilizable, evitando repetir código.

```bash
nombre_funcion() {
    local variable_local=$1   # $1 aquí es el primer argumento de la función, no del script
    comandos
    echo "resultado"          # "devolver" un valor = imprimirlo y capturarlo con $()
}

resultado=$(nombre_funcion "argumento")
```

`local` restringe la variable al cuerpo de la función — buena práctica para no "contaminar" el resto del script.

In [ ]:
%%bash
cd ~/curso_bioinfo/clase2/datos

cat > contar_lecturas.sh << 'EOF'
#!/bin/bash
set -euo pipefail

contar_lecturas() {
    local archivo=$1
    local n
    n=$(grep -c "^@" "$archivo")
    echo "$n"
}

for f in *.fastq; do
    resultado=$(contar_lecturas "$f")
    echo "$f tiene $resultado lecturas"
done
EOF
chmod +x contar_lecturas.sh
./contar_lecturas.sh

muestra1.fastq tiene 2 lecturas


muestra2.fastq tiene 2 lecturas


muestra3.fastq tiene 2 lecturas


---
## 9. Introducción a `sed` y `awk`

Estos dos comandos son herramientas clásicas de Unix para procesar texto línea por línea — extremadamente útiles para archivos tabulares (TSV/CSV) y formatos bioinformáticos (FASTA, FASTQ, VCF, GTF, etc.).

### `sed`: edición de flujo (sustituciones principalmente)

```bash
sed 's/patrón/reemplazo/' archivo      # reemplaza la primera ocurrencia por línea
sed 's/patrón/reemplazo/g' archivo     # reemplaza todas las ocurrencias (global)
```

### `awk`: procesamiento por columnas

`awk` divide cada línea en campos (por defecto, separados por espacios/tabs) accesibles como `$1`, `$2`, …, y `$0` es la línea completa.

```bash
awk '{print $1}' archivo                  # imprime la primera columna
awk -F'\t' '{print $1, $3}' archivo       # separador explícito: tab
awk -F'\t' '$2 > 100' archivo             # imprime solo líneas donde la columna 2 es > 100
```

In [ ]:
%%bash
cd ~/curso_bioinfo/clase2

cat > expresion.tsv << 'EOF'
gen	muestra1	muestra2
BRCA1	120	98
TP53	340	310
EGFR	45	52
MYC	670	590
GAPDH	5200	5100
EOF

echo "--- sed: renombrar un gen ---"
sed 's/BRCA1/BRCA1_HUMAN/' expresion.tsv

echo "--- awk: imprimir solo el nombre del gen y muestra1 ---"
awk -F'\t' '{print $1, $2}' expresion.tsv

echo "--- awk: filtrar genes con conteo > 100 en muestra1 (conservando encabezado) ---"
awk -F'\t' 'NR==1 || $2 > 100' expresion.tsv

echo "--- Combinando con una tuberia: contar cuantos genes superan el umbral ---"
awk -F'\t' 'NR>1 && $2 > 100' expresion.tsv | wc -l

--- sed: renombrar un gen ---


gen	muestra1	muestra2
BRCA1_HUMAN	120	98
TP53	340	310
EGFR	45	52
MYC	670	590
GAPDH	5200	5100


--- awk: imprimir solo el nombre del gen y muestra1 ---
gen muestra1
BRCA1 120
TP53 340
EGFR 45
MYC 

670
GAPDH 5200


--- awk: filtrar genes con conteo > 100 en muestra1 (conservando encabezado) ---


gen	muestra1	muestra2
BRCA1	120	98
TP53	340	310
MYC	670	590
GAPDH	5200	5100


--- Combinando con una tuberia: contar cuantos genes superan el umbral ---


4


---
## 10. Compresión y archivado: `tar` y `gzip`

Los archivos crudos de bioinformática (FASTQ, BAM, VCF) suelen ocuparse comprimidos para ahorrar espacio y ancho de banda al transferirlos.

| Comando | Uso |
|---|---|
| `gzip archivo` | Comprime `archivo` → `archivo.gz` (reemplaza el original) |
| `gzip -k archivo` | Comprime pero **conserva** el original (`-k` = *keep*) |
| `gunzip archivo.gz` | Descomprime |
| `zcat archivo.gz` | Muestra contenido comprimido sin descomprimir a disco |
| `tar -czvf salida.tar.gz directorio/` | Empaqueta y comprime un directorio completo (`c`=crear, `z`=gzip, `v`=verboso, `f`=nombre de archivo) |
| `tar -xzvf archivo.tar.gz` | Extrae un `.tar.gz` |
| `tar -tzvf archivo.tar.gz` | Lista el contenido sin extraer |

In [ ]:
%%bash
cd ~/curso_bioinfo/clase2

echo "--- Empaquetando y comprimiendo el directorio datos/ ---"
tar -czvf datos.tar.gz datos/

echo "--- Tamaño del paquete comprimido ---"
ls -lh datos.tar.gz

echo "--- Comprimiendo un archivo individual (conservando el original) ---"
gzip -k expresion.tsv
ls -lh expresion.tsv expresion.tsv.gz

echo "--- Viendo el contenido comprimido sin descomprimir a disco ---"
zcat expresion.tsv.gz | head -n 3

--- Empaquetando y comprimiendo el directorio datos/ ---


datos/
datos/contar_lecturas.sh
datos/muestra2.fastq
datos/muestra1.fastq
datos/muestra3.fastq


--- Tamaño del paquete comprimido ---


-rw-r--r-- 1 stoic-keen-fermi stoic-keen-fermi 417 Aug 10 11:39 datos.tar.gz


--- Comprimiendo un archivo individual (conservando el original) ---


-rw-r--r-- 1 stoic-keen-fermi stoic-keen-fermi  87 Aug 10 11:39 expresion.tsv
-rw-r--r-- 1 stoic-kee

n-fermi stoic-keen-fermi 112 Aug 10 11:39 expresion.tsv.gz


--- Viendo el contenido comprimido sin descomprimir a disco ---


gen	muestra1	muestra2
BRCA1	120	98
TP53	340	310


---
## 11. Ejercicio integrador (25 min)

Trabajen en parejas. El objetivo es escribir **un solo script** llamado `resumen_muestras.sh` dentro de `~/curso_bioinfo/clase2/` que:

1. Reciba como argumento un umbral mínimo de lecturas (`$1`).
2. Recorra con un `for` todos los archivos `*.fastq` del directorio `datos/` (usen comodines).
3. Para cada archivo, use una **función** `contar_lecturas` (como la de la Sección 8) para obtener el número de lecturas.
4. Use un `if` para clasificar cada archivo como `OK` (si tiene al menos el umbral de lecturas) o `REVISAR` (si tiene menos).
5. Vaya guardando los resultados (nombre de archivo, número de lecturas, estado) en un archivo `resumen.tsv`, con una columna por dato — usen `>>` (append) dentro del bucle.
6. Al final del script, imprima con `awk` cuántos archivos quedaron como `OK` y cuántos como `REVISAR`.
7. Comprima el archivo `resumen.tsv` con `gzip -k`.

Denle permisos de ejecución con `chmod` y ejecútenlo con distintos valores de umbral para comprobar que el resultado cambia.

Usen la celda de abajo como borrador (o practiquen directamente en su terminal si ya tienen acceso SSH al clúster).

In [ ]:
%%bash
cd ~/curso_bioinfo/clase2

# Escriban aqui su script resumen_muestras.sh (pueden usar cat << 'EOF' ... EOF)
# y luego ejecutenlo, por ejemplo:
# ./resumen_muestras.sh 3

---
## 12. Cierre y resumen

### Comandos y conceptos vistos hoy

```
export, unset, alias, unalias, which, type, source
PATH, ~/.bashrc
* ? {} (comodines / globbing)
#!/bin/bash, chmod +x, set -euo pipefail
$0 $1 $# $@ $?
if / elif / else / fi, test, [ ]
for ... in ... do ... done
while [ ] do ... done
while read -r var; do ... done < archivo
funcion() { local var=...; }
sed 's/a/b/'
awk '{print $1}', awk -F'\t' '$2 > 100'
tar -czvf, tar -xzvf, gzip, gzip -k, gunzip, zcat
```

### Cheatsheet rápido

| Necesito... | Comando |
|---|---|
| Ver/definir una variable de entorno | `export VAR=valor`, `echo $VAR` |
| Ver dónde busca ejecutables el sistema | `echo $PATH` |
| Crear un atajo de comando | `alias nombre='comando'` |
| Hacer permanente una config | Agregarla a `~/.bashrc` y `source ~/.bashrc` |
| Referirme a varios archivos a la vez | `*.ext`, `archivo?.ext`, `*.{ext1,ext2}` |
| Hacer ejecutable un script | `chmod +x script.sh` |
| Leer un argumento del script | `$1`, `$2`, `$#`, `$@` |
| Evaluar una condición | `if [ cond ]; then ... fi` |
| Repetir sobre una lista de archivos | `for f in *.ext; do ... done` |
| Repetir mientras algo sea cierto | `while [ cond ]; do ... done` |
| Recorrer un archivo línea por línea | `while read -r linea; do ... done < archivo` |
| Reutilizar código | `funcion() { ...; }` |
| Sustituir texto en un archivo | `sed 's/a/b/g' archivo` |
| Procesar columnas de una tabla | `awk -F'\t' '{print $1}' archivo` |
| Comprimir/empaquetar | `gzip -k archivo`, `tar -czvf out.tar.gz dir/` |

### Para la próxima clase (Clase 3 — Manejo de archivos en la nube y configuración de ambientes bioinformáticos)

- Practicar los scripts de hoy en su propia sesión SSH al clúster (no solo en este notebook).
- Repasar `for`, `while` e `if`: la próxima clase los usaremos para automatizar la descarga y organización de archivos.
- Se revisará transferencia de archivos (`scp`, `rsync`) y gestores de ambientes/paquetes (`conda`/`mamba`) — no requiere preparación previa, pero quienes quieran adelantar pueden leer sobre qué es un *environment* de conda.

### Recursos adicionales

- `man bash` — manual completo de bash.
- `help if`, `help for`, `help test` — ayuda de comandos internos de bash.
- [The Linux Command Line (libro gratuito, W. Shotts)](https://linuxcommand.org/tlcl.php) — capítulos 24-36 (scripting).
- [Software Carpentry — The Unix Shell: Loops y Shell Scripts](https://swcarpentry.github.io/shell-novice/)
- [ShellCheck](https://www.shellcheck.net/) — herramienta online para detectar errores comunes en scripts de bash.